# Notebook 2 — Clustering & Validation

> Entraînement MiniBatchKMeans, sélection du K optimal par métriques, validation et sauvegarde du modèle.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)


In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 2.1 Preprocessing

Normalisation et sélection des features pour le clustering.


### Sélection des features

Exclure les features trop corrélées ou peu discriminantes.


In [ ]:

import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import DATA_PATH, MODELS_PATH
from src.clustering import (CLUSTERING_FEATURES, preprocess, find_optimal_k,
                              train_model, validate_model, save_model, load_model,
                              plot_elbow_curve, plot_clusters_pca)

# Chargement des features train (Jan–Juin uniquement)
feat_train = pd.read_csv(os.path.join(DATA_PATH, "customer_features_train.csv"),
                          index_col="anonymized_card_code")
print(f"Features train : {feat_train.shape}")

# Colonnes disponibles pour le clustering
clust_cols = [c for c in CLUSTERING_FEATURES if c in feat_train.columns]
print(f"\nFeatures de clustering ({len(clust_cols)}) :")
print(clust_cols)


### StandardScaler

Normalisation obligatoire — mean=0, std=1. Ajustement sur train uniquement.


In [ ]:

# StandardScaler — fit sur train uniquement
X_train, scaler, cols_used = preprocess(feat_train, fit=True)
print(f"Matrice X_train : {X_train.shape}")
print(f"Features utilisées : {cols_used}")
print(f"\nMoyennes scaler (premières 5) : {scaler.mean_[:5].round(3)}")
print(f"Stddev scaler (premières 5) : {scaler.scale_[:5].round(3)}")


### Sauvegarde du scaler

models/scaler.pkl — indispensable pour la simulation dynamique.


In [ ]:

import joblib, os
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
print("✓ Scaler sauvegardé → models/scaler.pkl")
print(f"  n_features_in_ = {scaler.n_features_in_}")


---

## 2.2 Sélection du K optimal

Comparaison de K=4 à K=11 avec 3 métriques.


### Elbow method (Inertia)

Courbe inertie en fonction de K — chercher le coude.


In [ ]:

# Elbow method + métriques sur K=4→11
# ⚠️ Peut prendre 5-15 minutes selon la taille du dataset
print("Itération K=4 à K=11 — calcul Silhouette, Calinski-Harabasz, Inertia...")
k_results = find_optimal_k(X_train)
print(f"\n✓ K optimal retenu : {k_results['optimal_k']}")


### Silhouette Score

Mesure la qualité de séparation des clusters (objectif > 0.40).


In [ ]:

# Tableau récap des scores par K
scores_df = pd.DataFrame({
    "K":                k_results["k_values"],
    "Inertia":          [f"{v:,.0f}" for v in k_results["inertias"]],
    "Silhouette":       [f"{v:.4f}" for v in k_results["silhouette_scores"]],
    "Calinski-Harabasz": [f"{v:,.0f}" for v in k_results["calinski_scores"]],
    "Davies-Bouldin":   [f"{v:.4f}" for v in k_results["davies_scores"]],
}).set_index("K")

# Marquer le K optimal
scores_df.index = [f"K={k} ← OPTIMAL" if k == k_results["optimal_k"]
                   else f"K={k}" for k in k_results["k_values"]]
print("=== COMPARAISON MÉTRIQUES K=4 à K=11 ===")
print(scores_df.to_string())


### Calinski-Harabasz Index

Ratio inter/intra-cluster — maximiser.


In [ ]:

# Calinski-Harabasz : plus élevé = meilleure séparation inter-cluster
print(f"Calinski-Harabasz max à K={k_results['k_values'][int(np.argmax(k_results['calinski_scores']))]}")
print(f"Davies-Bouldin min  à K={k_results['k_values'][int(np.argmin(k_results['davies_scores']))]}")
print(f"Silhouette max      à K={k_results['k_values'][int(np.argmax(k_results['silhouette_scores']))]}")
print(f"\n→ K retenu = {k_results['optimal_k']} (consensus 3 métriques + interprétabilité business)")


### Choix final de K

Synthèse des 3 métriques + justification business du choix.


In [ ]:

# Visualisation elbow + silhouette
path = plot_elbow_curve(k_results)
print(f"✓ Courbes sauvegardées → {path}")

# Afficher inline
img = plt.imread(path)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(img)
ax.axis("off")
plt.show()


---

## 2.3 Entraînement du modèle final

MiniBatchKMeans avec le K optimal.


### Entraînement

MiniBatchKMeans(n_clusters=K, random_state=42, n_init=10).fit(X_train_scaled).


In [ ]:

# Entraînement avec K optimal
K_OPTIMAL = k_results["optimal_k"]
model = train_model(X_train, k=K_OPTIMAL)
print(f"\n✓ Modèle entraîné : K={K_OPTIMAL}, Inertia={model.inertia_:.0f}")


### Assignation des clusters

Prédiction sur le dataset complet.


In [ ]:

# Assignation des clusters sur train
labels_train = model.predict(X_train)
feat_train["cluster"] = labels_train

# Distribution
dist = pd.Series(labels_train).value_counts().sort_index()
ca_by_cluster = feat_train.groupby("cluster")["monetary"].sum()

print("=== DISTRIBUTION DES CLUSTERS ===")
for cl in sorted(dist.index):
    n = dist[cl]
    ca = ca_by_cluster.get(cl, 0)
    print(f"  Cluster {cl} : {n:6,} clients ({n/len(labels_train)*100:5.1f}%) "
          f"| CA total {ca:>12,.0f} € ({ca/feat_train['monetary'].sum()*100:.1f}%)")

print(f"\nTotal : {len(labels_train):,} clients")


### Sauvegarde

models/kmeans_model.pkl via joblib.


In [ ]:

# Sauvegarde modèle + scaler + features avec cluster
paths = save_model(model, scaler)
print(f"\n✓ Modèle → {paths['model']}")
print(f"✓ Scaler → {paths['scaler']}")

# Sauvegarder les features avec cluster pour les notebooks suivants
feat_train.to_csv("../data/customer_features_train.csv")
print("✓ Features train avec cluster → data/customer_features_train.csv")


---

## 2.4 Visualisation des clusters

UMAP 2D et distributions par cluster.


### Réduction dimensionnelle UMAP

Projection 2D pour visualiser la séparation des clusters.


In [ ]:

# PCA 2D (UMAP nécessite umap-learn optionnel — fallback sur PCA)
path_pca = plot_clusters_pca(X_train, labels_train)
print(f"✓ Visualisation clusters → {path_pca}")

img = plt.imread(path_pca)
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(img)
ax.axis("off")
plt.show()


### Distribution des tailles

Nombre de clients et part de CA par cluster.


In [ ]:

# Distribution taille + CA par cluster — graphique
fig, axes_c = plt.subplots(1, 2, figsize=(12, 5))
cluster_ids = sorted(dist.index)
colors_c = plt.cm.RdPu(np.linspace(0.2, 0.9, len(cluster_ids)))

# Tailles
axes_c[0].bar(cluster_ids, [dist[c] for c in cluster_ids], color=colors_c, edgecolor="white", alpha=0.9)
axes_c[0].set_title("Nb clients par cluster", fontweight="bold")
axes_c[0].set_xlabel("Cluster")
axes_c[0].set_ylabel("Nb clients")
for i, c in enumerate(cluster_ids):
    axes_c[0].text(c, dist[c] + 50, f"{dist[c]:,}", ha="center", va="bottom", fontsize=9)

# CA
ca_vals = [ca_by_cluster.get(c, 0) / 1000 for c in cluster_ids]
axes_c[1].bar(cluster_ids, ca_vals, color=colors_c, edgecolor="white", alpha=0.9)
axes_c[1].set_title("CA total par cluster (k€)", fontweight="bold")
axes_c[1].set_xlabel("Cluster")
axes_c[1].set_ylabel("CA (k€)")
for i, c in enumerate(cluster_ids):
    axes_c[1].text(c, ca_vals[i] + 1, f"{ca_vals[i]:.0f}k€", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
plt.savefig("../outputs/figures/2_cluster_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


---

## 2.5 Validation statistique

Tests statistiques pour confirmer la significativité des clusters.


### ANOVA / Kruskal-Wallis

Confirmer que les features diffèrent significativement entre clusters.


In [ ]:

from scipy.stats import kruskal

# Test Kruskal-Wallis pour chaque feature de clustering
print("=== TESTS KRUSKAL-WALLIS (H₀ : distributions identiques entre clusters) ===")
print(f"{'Feature':<35} {'H-statistic':>12} {'p-value':>12} {'Significatif?':>15}")
print("-" * 80)

kw_results = []
for col in cols_used[:15]:  # tester les 15 premières features
    groups = [feat_train[feat_train["cluster"] == c][col].dropna().values
              for c in sorted(feat_train["cluster"].unique())]
    if all(len(g) > 5 for g in groups):
        h_stat, p_val = kruskal(*groups)
        sig = "✓ OUI" if p_val < 0.05 else "✗ NON"
        kw_results.append({"feature": col, "H": h_stat, "p": p_val})
        print(f"{col:<35} {h_stat:>12.1f} {p_val:>12.2e} {sig:>15}")

print(f"\n→ Tous les K-W significatifs = clusters bien séparés statistiquement")


### Stabilité bootstrap

Rééchantillonner 80% et mesurer la cohérence des assignations (ARI).


In [ ]:

from sklearn.metrics import adjusted_rand_score

# Bootstrap stability : rééchantillonner 80% des clients, refitter, comparer
print("=== STABILITÉ BOOTSTRAP (5 iterations × 80% des clients) ===")
n_bootstrap = 5
ari_scores = []

for i in range(n_bootstrap):
    np.random.seed(RANDOM_SEED + i)
    idx = np.random.choice(len(X_train), size=int(0.8 * len(X_train)), replace=False)
    X_sub = X_train[idx]

    model_boot = train_model(X_sub, k=K_OPTIMAL, random_state=RANDOM_SEED + i)
    labels_sub = model_boot.predict(X_sub)
    labels_orig = model.predict(X_sub)

    ari = adjusted_rand_score(labels_orig, labels_sub)
    ari_scores.append(ari)
    print(f"  Bootstrap {i+1} : ARI = {ari:.4f}")

print(f"\n→ ARI moyen : {np.mean(ari_scores):.4f} ± {np.std(ari_scores):.4f}")
print(f"  ARI > 0.80 = clusters très stables | ARI > 0.60 = acceptables")

# Rapport de validation final
report = validate_model(X_train, labels_train)
